In [11]:
import matplotlib.pyplot as plt
import numpy as np

def noise_level(t_gates, data_val, noise_slope_exponent=-0.5, noise_level_at_1ms=1e-8, moment=1, relative_noise=0.03):
    b = noise_level_at_1ms/(1e-3**noise_slope_exponent)
    abs_noise = b*(t_gates**noise_slope_exponent)
    y = data_val*moment
    std_rel = abs_noise/y
    std_tot = np.sqrt(std_rel**2 + relative_noise**2)
    
    y = np.abs(y)
    ylb = np.abs(y/(std_tot+1)-y)
    yub = np.abs(y*(std_tot+1)-y)

    fig,axs = plt.subplots()
    axs.loglog(t_gates, abs_noise, 'r-', linewidth=0.5)
    # axs.loglog(t_gates, y, 'k-', linewidth=0.5)
    axs.errorbar(t_gates, y, yerr=(ylb, yub), ecolor='k')
    axs.set_aspect('equal')
    plt.show()

    return y, ylb, yub

In [12]:
%matplotlib qt5
t_gates = np.geomspace(1e-5, 1e-2, 31)
dbdt = 1e-20*t_gates**(-5/2)
y,ylb,yub = noise_level(t_gates, dbdt, moment=300*40*4)

In [9]:
def cal_std_tem(t_h, level_a, std_uni, g_time, g_data, moment):
    mm = 1

    if mm == 1:
        g_data_m = g_data * moment  # LM HM in separate files moment=amp*turns (not *areal)
    elif mm == 2:  # LM and HM times in the same tem file
        # g_data[:6] = g_data[:6] * (40 * 1)  # 40amp * 1 turn LM
        # g_data[6:] = g_data[6:] * (95 * 4)  # 95amp * 4 turn HM
        pass

    b = level_a / (1e-3**t_h)
    abs_a = b * (g_time**t_h)

    abs_noise = abs_a  # + abs_b  # total absolute noise
    std_rel = abs_noise / g_data_m  # relative absolute noise
    std_tot = np.sqrt(std_uni**2 + std_rel**2)  # including uniform std

    g_dat = np.abs(g_data_m)

    plt.plot(g_time, g_dat)
    plt.plot(g_time, abs_a, '-g')
    plt.errorbar(g_time, g_dat, yerr=(np.abs(g_dat / (std_tot + 1) - g_dat), np.abs(g_dat - (g_dat * (std_tot + 1)))), fmt='r')
    plt.yscale('log')
    plt.xscale('log')
    plt.grid(True)
    plt.axis([1e-5, 1e-2, 1e-11, 1e-2])
    plt.show()

In [10]:
cal_std_tem(-1/2, 1e-5, 1e-5, t_gates, dbdt, 300*40*4)